In [2]:
import os

# 1. Route all Hugging Face downloads to your spacious persistent volume
os.environ["HF_HOME"] = "/workspace/huggingface_cache"

# 2. Keep the fix for the Xet network timeout issue
os.environ["HF_HUB_DISABLE_XET"] = "1"

print(f"Hugging Face cache set to: {os.environ['HF_HOME']}")

Hugging Face cache set to: /workspace/huggingface_cache


In [ ]:
import torch
from unsloth import FastVisionModel
from tqdm import tqdm
import PIL.Image

# 1. Load the Base Model in 4-bit Precision
# Unsloth handles the memory optimization automatically
model, processor = FastVisionModel.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct",
    load_in_4bit=True,  # CRITICAL: This allows it to fit on your RTX 3050
    use_gradient_checkpointing="unsloth",
)

# 2. Load your LoRA Adapter
# Point this to your trained SFT adapter weights
model.load_adapter("/workspace/ImageToSpec_Stage1/qwen3b_lora_sft_bf16_final")

# 3. Optimize for Inference
FastVisionModel.for_inference(model)

# Ensure image resolution doesn't blow up VRAM
processor.image_processor.max_pixels = 262144  # Limit to 512x512 max resolution

all_completions = []
all_targets = []

print("Running Local 4-bit Inference...")

# Note: We use a batch size of 1 here because your 4GB/6GB VRAM
# cannot handle processing multiple images simultaneously.
for item in tqdm(eval_dataset):
    messages = item["messages"]

    # We only feed the user prompt to the model (messages[0])
    prompt = processor.apply_chat_template(
        [messages[0]], tokenize=False, add_generation_prompt=True)

    # Load the image using PIL
    image_path = item["images"][0]
    pil_image = PIL.Image.open(image_path).convert("RGB")

    # Format inputs for Qwen-VL
    inputs = processor(
        text=[prompt],
        images=[pil_image],
        return_tensors="pt",
        padding=True
    ).to("cuda")

    # Generate completion
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=1500,  # Keep this reasonable for your JSON specs
            temperature=0.0      # Greedy decoding for consistent evaluation
        )

    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    completion = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    all_completions.append(completion)
    all_targets.append(item["target_specs"])


# ==========================================
# 3. Score the completions
# ==========================================
print("\n=== EVALUATION RESULTS ===")

format_scores = format_reward_func(all_completions)
print(f"Format Accuracy:         {sum(1 for s in format_scores if s > 0) / len(format_scores) * 100:.2f}%")

schema_scores = schema_enforcement_reward_func(all_completions)
print(f"Avg Schema Reward:       {sum(schema_scores)/len(schema_scores):.4f}")

arch_scores = panel_architecture_reward_func(all_completions)
print(f"Avg Architecture Reward: {sum(arch_scores)/len(arch_scores):.4f}")

axis_scores = axis_mapping_reward_func(all_completions)
print(f"Avg Axis Reward:         {sum(axis_scores)/len(axis_scores):.4f}")

data_scores = dynamic_series_reward_func(all_completions, all_targets)
print(f"Avg Series Data Reward:  {sum(data_scores)/len(data_scores):.4f}")

RuntimeError: The NVIDIA driver on your system is too old (found version 12040). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver.